# Chest X-Ray Disease Classification - Medical Classification Model
This notebook details the end-to-end Machine Learning pipeline using transfer learning with **DenseNet121** to classify medical scans.

### Workflow Steps:
1. **Dataset Discovery & Analysis**: Discover classes, image counts, and check for class imbalance.
2. **Data Preprocessing & Splitting**: Stratified split of 70% Train, 15% Validation, 15% Test.
3. **Data Augmentation**: Applying horizontal flip, rotation, and zoom to reduce overfitting.
4. **Transfer Learning**: Instantiate DenseNet121 with ImageNet weights, freeze base layers, and add a custom classification head.
5. **Model Training**: Train the custom head using class weights.
6. **Evaluation**: Evaluate on the test split, generate accuracy/loss graphs, classification reports, and confusion matrices.
7. **Interpretability with Grad-CAM**: Generate heatmaps highlighting features that influenced predictions.
8. **Export**: Save the final trained model for production inference.

## 1. Environment Setup & Imports

In [ ]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight

# Set random seeds for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

print("Tensorflow version:", tf.__version__)
print("GPU Available:", tf.config.list_physical_devices('GPU'))

## 2. Dataset Loading & Class Distribution Analysis

In [ ]:
dataset_dir = Path("datasets/xray")
classes = ['COVID19', 'NORMAL', 'PNEUMONIA']

all_paths = []
all_labels = []

for label_idx, class_name in enumerate(classes):
    # Iterate through all subfolders dynamically
    for root, dirs, files in os.walk(dataset_dir):
        if class_name in root:
            for f in files:
                if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp')):
                    all_paths.append(os.path.join(root, f))
                    all_labels.append(label_idx)

all_paths = np.array(all_paths)
all_labels = np.array(all_labels)

print(f"Total images found: {len(all_paths)}")
for i, name in enumerate(classes):
    count = np.sum(all_labels == i)
    print(f"  - {name}: {count} images ({count/len(all_paths)*100:.1f}%)")

In [ ]:
plt.figure(figsize=(8, 5))
counts = [np.sum(all_labels == i) for i in range(len(classes))]
plt.bar(classes, counts, color=['#3b82f6', '#10b981', '#f59e0b', '#ef4444'][:len(classes)])
plt.title('Class Distribution')
plt.xlabel('Diagnostic Class')
plt.ylabel('Image Count')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()

## 3. Visualize Sample Images

In [ ]:
plt.figure(figsize=(12, 6))
for idx, class_name in enumerate(classes):
    class_indices = np.where(all_labels == idx)[0]
    random_idx = np.random.choice(class_indices)
    img_path = all_paths[random_idx]
    
    img = cv2.imread(img_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    
    plt.subplot(1, len(classes), idx + 1)
    plt.imshow(img)
    plt.title(f"{class_name}\nShape: {img.shape}")
    plt.axis('off')
plt.tight_layout()
plt.show()

## 4. Stratified Train/Val/Test Split (70/15/15)

In [ ]:
train_paths, temp_paths, train_labels, temp_labels = train_test_split(
    all_paths, all_labels, test_size=0.30, random_state=42, stratify=all_labels
)
val_paths, test_paths, val_labels, test_labels = train_test_split(
    temp_paths, temp_labels, test_size=0.50, random_state=42, stratify=temp_labels
)

print(f"Train set size: {len(train_paths)}")
print(f"Validation set size: {len(val_paths)}")
print(f"Test set size: {len(test_paths)}")

## 5. Input Pipeline & Data Augmentation

In [ ]:
IMG_SIZE = (224, 224)
BATCH_SIZE = 32

def parse_image(file_path, label):
    img = tf.io.read_file(file_path)
    img = tf.image.decode_image(img, channels=3, expand_animations=False)
    img.set_shape([None, None, 3])
    img = tf.image.resize(img, IMG_SIZE)
    img = tf.keras.applications.densenet.preprocess_input(img)
    return img, label

data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.1),
    tf.keras.layers.RandomZoom(0.1),
])

train_ds = tf.data.Dataset.from_tensor_slices((train_paths, train_labels))
train_ds = train_ds.shuffle(buffer_size=1000).map(parse_image, num_parallel_calls=tf.data.AUTOTUNE)
train_ds = train_ds.batch(BATCH_SIZE).map(lambda x, y: (data_augmentation(x, training=True), y), num_parallel_calls=tf.data.AUTOTUNE).prefetch(buffer_size=tf.data.AUTOTUNE)

val_ds = tf.data.Dataset.from_tensor_slices((val_paths, val_labels))
val_ds = val_ds.map(parse_image, num_parallel_calls=tf.data.AUTOTUNE).batch(BATCH_SIZE).prefetch(buffer_size=tf.data.AUTOTUNE)

test_ds = tf.data.Dataset.from_tensor_slices((test_paths, test_labels))
test_ds = test_ds.map(parse_image, num_parallel_calls=tf.data.AUTOTUNE).batch(BATCH_SIZE).prefetch(buffer_size=tf.data.AUTOTUNE)

class_weights = compute_class_weight('balanced', classes=np.unique(train_labels), y=train_labels)
class_weights_dict = {i: w for i, w in enumerate(class_weights)}
print("Class weights dict:", class_weights_dict)

## 6. DenseNet121 Transfer Learning Model Setup

In [ ]:
base_model = tf.keras.applications.DenseNet121(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
base_model.trainable = False  # Freeze pretrained weights

inputs = tf.keras.Input(shape=(224, 224, 3))
x = base_model(inputs, training=False)
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dense(256, activation='relu')(x)
x = tf.keras.layers.Dropout(0.5)(x)
outputs = tf.keras.layers.Dense(len(classes), activation='softmax')(x)

model = tf.keras.Model(inputs, outputs)
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)
model.summary()

## 7. Model Training

In [ ]:
checkpoint_callback = tf.keras.callbacks.ModelCheckpoint(
    filepath="models/xray_model.keras",
    monitor="val_accuracy",
    save_best_only=True,
    verbose=1
)
early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=2,
    restore_best_weights=True,
    verbose=1
)

history = model.fit(
    train_ds,
    epochs=3,
    validation_data=val_ds,
    class_weight=class_weights_dict,
    callbacks=[checkpoint_callback, early_stopping]
)

## 8. Training History Plots

In [ ]:
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Train Accuracy', color='#2563eb')
plt.plot(history.history['val_accuracy'], label='Val Accuracy', color='#10b981')
plt.title('Accuracy curves')
plt.legend()
plt.grid(True)

plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Train Loss', color='#2563eb')
plt.plot(history.history['val_loss'], label='Val Loss', color='#10b981')
plt.title('Loss curves')
plt.legend()
plt.grid(True)
plt.show()

## 9. Model Evaluation

In [ ]:
test_loss, test_acc = model.evaluate(test_ds, verbose=0)
print(f"Test Accuracy: {test_acc:.4f}")
print(f"Test Loss: {test_loss:.4f}")

predictions = []
true_labels = []
for imgs, lbls in test_ds:
    preds = model.predict(imgs, verbose=0)
    predictions.extend(np.argmax(preds, axis=1))
    true_labels.extend(lbls.numpy())

predictions = np.array(predictions)
true_labels = np.array(true_labels)

print("Classification Report:")
print(classification_report(true_labels, predictions, target_names=classes))
print("Confusion Matrix:")
print(confusion_matrix(true_labels, predictions))

## 10. Grad-CAM Visualization

In [ ]:
from utils.gradcam import generate_gradcam

# Get a sample image from the test set
sample_path = test_paths[0]
actual_class = classes[test_labels[0]]

gradcam_img, pred_idx, confidence = generate_gradcam(sample_path, model, target_size=(224, 224))
pred_class = classes[pred_idx]

orig_img = cv2.imread(sample_path)
orig_img = cv2.cvtColor(orig_img, cv2.COLOR_BGR2RGB)
gradcam_rgb = cv2.cvtColor(gradcam_img, cv2.COLOR_BGR2RGB)

plt.figure(figsize=(10, 5))
plt.subplot(1, 2, 1)
plt.imshow(orig_img)
plt.title(f"Original Scan\n(Class: {actual_class})")
plt.axis('off')

plt.subplot(1, 2, 2)
plt.imshow(gradcam_rgb)
plt.title(f"Grad-CAM Heatmap\n(Pred: {pred_class} | Conf: {confidence*100:.1f}%)")
plt.axis('off')
plt.show()